# MedSAM2 — pure C++ on GPU (Colab)

Dependency-free C++ port of **MedSAM2** (SAM2 / Hiera fine-tuned on medical 3D volumes & videos). No
PyTorch at run time: the **Hiera** image encoder + **SAM2** mask decoder are compiled with
`nvcc -DUSE_CUDA` so the matmul/conv GEMMs offload to **cuBLAS** via the `bk::gemm_hosted` seam.
Runtime → Change runtime type → **GPU** (T4 is enough).

Same recipe as the sibling `medsam_cpp` GPU notebook — only the architecture (Hiera + SAM2 decoder)
and the weight-export differ.

## 1. GPU + repo

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2

In [ ]:
%cd /content
![ -d medsam2_cpp ] || git clone https://github.com/yomei-o/medsam2_cpp.git
%cd /content/medsam2_cpp

## 2. Weights (one-time extraction — the only Python step)
Installs the `sam2` package (needs torch ≥ 2.5.1; `SAM2_BUILD_CUDA=0` skips the custom CUDA ext),
downloads the **MedSAM2_latest.pt** checkpoint from Hugging Face (`wanglab/MedSAM2`), then runs the
export scripts that dump the Hiera encoder + SAM2 decoder weights into `pure/ref/*.bin`.

In [ ]:
import os
os.environ['SAM2_BUILD_CUDA'] = '0'
!pip -q install 'torch>=2.5.1' git+https://github.com/facebookresearch/sam2.git huggingface_hub numpy
from huggingface_hub import hf_hub_download
import shutil, glob
os.makedirs('pure/ref', exist_ok=True)
ckpt = hf_hub_download('wanglab/MedSAM2', 'MedSAM2_latest.pt')
shutil.copy(ckpt, 'pure/ref/MedSAM2_latest.pt')
print('checkpoint:', ckpt)
# extract Hiera encoder + SAM2 decoder (also writes dec_emb/hr256/hr128 used by training)
!cd pure/ref && python export_hiera.py && python export_medsam2_dec.py
# (optional — for the full 3D/video track: also run export_memattn.py export_memenc.py export_prop.py)
print('bins:', sorted(os.path.basename(p) for p in glob.glob('pure/ref/*weights*.bin')))

## 3. Build — CPU (g++, Eigen) and GPU (cuBLAS via nvcc -DUSE_CUDA)

In [ ]:
# CPU build: Eigen (blocked+SIMD GEMM) + OpenMP -> the Hiera encode is far faster than plain scalar -O2.
# GPU build: nvcc -DUSE_CUDA offloads GEMMs to cuBLAS. C++20 (the Hiera/propagation code uses it).
!g++ -O3 -std=c++20 -march=native -fopenmp -DUSE_EIGEN -DNOMINMAX -Ipure/third_party -Ipure/third_party/eigen_flat pure/infer_medsam2.cpp -o infer_cpu
!nvcc -x cu -O2 -std=c++20 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/infer_medsam2.cpp -lcublas -o infer_gpu
print('built infer_cpu (Eigen+OpenMP) / infer_gpu (cuBLAS)')

## 4. Segment a medical image (point prompt) — CPU vs GPU

In [ ]:
import urllib.request, time, os
urllib.request.urlretrieve('https://raw.githubusercontent.com/bowang-lab/MedSAM/main/assets/img_demo.png', 'med.png')
for n in ['infer_cpu', 'infer_gpu']:
    t = time.time(); os.system(f'./{n} med.png 256 256 out_{n}.png pure/ref'); print(f'{n:10s} {time.time()-t:5.1f}s')

In [ ]:
from IPython.display import Image, display
display(Image('med.png', width=320), Image('out_infer_gpu.png', width=320))

## 5. Fine-tune the SAM2 mask decoder on GPU (synthetic)
Freezes the Hiera encoder, trains the decoder on a real frozen embedding + a synthetic target mask with
focal+dice (+ IoU MSE) — the `mask_loss` must drop. This is MedSAM2's `finetune_sam2_img.py` recipe.

In [ ]:
!nvcc -x cu -O2 -std=c++20 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Xcompiler -fopenmp -Ipure/third_party pure/train_medsam2.cpp -lcublas -o train_gpu
!./train_gpu pure/ref --steps 20